В этом задании вам предстоит дообучить трансформерную модель для задачи классификации с помощью различных техник и сравнить их между собой.

Датасет: **dair-ai/emotion**

Модель: **google-bert/bert-base-uncased** (если хочется, можно заменить на что-то более интересное)

-----------------

Скачайте датасет и модель. Измерьте базовые метрики классификации перед началом экспериментов.
**NB!** Для всех типов дообучения замерьте :

1. качество классификации на выходе
2. время дообучения
3. количество параметров для обучения
4. потребление ресурсов (не нужно заморачиваться с профайлингом - можно просто посмотреть в nvidia-smi или torch.cuda.memory_allocated)
-------------------

1. Обучите модель в режиме full finetuning - 1 балл

2. Обучите модель в режиме linear probing - реализуйте кастомную классификационную голову и обучайте только ее. Не забудьте описать, чем обусловлено устройство головы, как вы пришли к такой архитектуре - 2 балла

3. Обучите модель в режиме PEFT с использованием prompt tuning или prefix tuning. При выборе метода напишите пару слов, почему решили остановиться именно на этом методе - 2 балла

4. Обучите модель в режиме PEFT с использованием LoRA. Попробуйте подобрать оптимальный ранг - r, при желании поэкспериментируйте с остальными гиперпараметрами. Опишите, чем обусловлена ваша финальная конфигурация - 2 балла

5. Соберите все результаты отдельных замеров в таблицу и сделайте выводы о вычислительной сложности методов, итоговом качестве и прочих наблюдаемых свойствах моделей - 1 балл

**Общее**:
1. Принимаемые решения обоснованы (почему выбрана определенная архитектура/гиперпараметр/оптимизатор/преобразование и т.п.) - 1 балл
2. Обеспечена воспроизводимость решения: зафиксированы random_state, ноутбук воспроизводится от начала до конца без ошибок - 1 балл

In [ ]:
%%capture 
!pip install -q transformers datasets evaluate peft

In [3]:
import torch
import numpy as np
import random
import evaluate
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, EarlyStoppingCallback
from peft import get_peft_model, LoraConfig, TaskType, PromptTuningConfig, PrefixTuningConfig, PromptTuningInit
from sklearn.metrics import accuracy_score, f1_score
import time
import pandas as pd

2025-04-24 19:22:32.905345: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1745522553.344679      31 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1745522553.472867      31 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [4]:
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

MODEL_NAME = "bert-base-uncased"
BATCH_SIZE = 16
EPOCHS = 3
MAX_LENGTH = 128
NUM_LABELS = 6

Загрузка датасета

In [5]:
dataset = load_dataset("dair-ai/emotion")

README.md:   0%|          | 0.00/9.05k [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/1.03M [00:00<?, ?B/s]

validation-00000-of-00001.parquet:   0%|          | 0.00/127k [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/129k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/16000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [6]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [7]:
def preprocess(example):
    return tokenizer(example["text"], truncation=True, padding="max_length", max_length=MAX_LENGTH)

tokenized_datasets = dataset.map(preprocess)

Map:   0%|          | 0/16000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [8]:
id2label = dataset["train"].features["label"].int2str
label2id = dataset["train"].features["label"].str2int

Будем дополнительно выводить метрику точности и F1-Score

In [ ]:
def compute_metrics(pred):
    labels = pred.label_ids
    preds = np.argmax(pred.predictions, axis=1)
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average="macro")
    return {"accuracy": acc, "f1": f1}

In [9]:
def train_model(model, train_data, eval_data, name, peft_cfg=None):
    if peft_cfg:
        model = get_peft_model(model, peft_cfg)
    
    training_args = TrainingArguments(
        output_dir=f"./results/{name}",
        eval_strategy="epoch",
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        num_train_epochs=EPOCHS,
        seed=seed,
        save_strategy="no",
        logging_dir=f"./logs/{name}",
        report_to="none"
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_data,
        eval_dataset=eval_data,
        compute_metrics=compute_metrics
    )

    start = time.time()
    trainer.train()
    end = time.time()

    results = trainer.evaluate()
    results.update({
        "trainable_params": sum(p.numel() for p in model.parameters() if p.requires_grad),  # общее кол-во параметров
        "total_params": sum(p.numel() for p in model.parameters()),                         # кол-во обучаемых параметров
        "training_time_min": (end - start) / 60,                                            # время обучения
        "gpu_memory_mb": torch.cuda.max_memory_allocated() / 1024**2                        # потребление ресурсов
    })
    return results

### Обучение модели в режиме full finetuning

Обучаются все параметры модели

In [21]:
model_full = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=NUM_LABELS)
results_full = train_model(model_full, tokenized_datasets["train"], tokenized_datasets["validation"], name="full_finetuning")

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.433100,0.162544,0.927500,0.905415
2,0.125900,0.144097,0.938500,0.912406
3,0.080100,0.150185,0.939000,0.911826


/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


### Обучение модели в режиме linear probing 

Linear -> ReLU -> Dropout -> Linear — это типичная архитектура небольшой feedforward-сети, которая:

- повышает обучаемость и экспрессивность модели
- позволяет лучше захватывать сложные зависимости в представлениях
- Dropout служит для регуляризации, чтобы избежать переобучения (важно при маленьком количестве обучаемых параметров)



Обучается новая классификационная голова, написанная ниже

In [16]:
class LinearProbeHead(torch.nn.Module):
    def __init__(self, hidden_size, num_labels):
        super().__init__()
        self.classifier = torch.nn.Sequential(
            torch.nn.Linear(hidden_size, hidden_size),
            torch.nn.ReLU(),
            torch.nn.Dropout(0.1),
            torch.nn.Linear(hidden_size, num_labels)
        )

    def forward(self, x):
        return self.classifier(x)

base_model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=NUM_LABELS)
for param in base_model.bert.parameters():
    param.requires_grad = False

results_linear = train_model(base_model, tokenized_datasets["train"], tokenized_datasets["validation"], name="linear_probing")


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,1.585100,1.562322,0.373000,0.123864
2,1.561600,1.555506,0.386500,0.144113
3,1.554700,1.554026,0.393000,0.150307


/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


### Обучение модели в режиме PEFT с использованием prompt tuning или prefix tuning. 


1. Prefix Tuning позволяет достичь почти того же качества, что и full finetuning, но при этом обновляет меньше параметров (только виртуальные токены).
2. В отличие от prompt tuning (который более эффективен для больших моделей и генерации), prefix tuning отлично работает в задачах sequence classification (emotion — классификация эмоций по тексту).

In [18]:
prefix_config = PrefixTuningConfig(
    task_type=TaskType.SEQ_CLS,
    num_virtual_tokens=20,
    num_layers=12,
    encoder_hidden_size=768
)

model_prefix = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=NUM_LABELS)
results_prefix = train_model(model_prefix, tokenized_datasets["train"], tokenized_datasets["validation"], name="prefix_tuning", peft_cfg=prefix_config)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
No label_names provided for model class `PeftModelForSequenceClassification`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,1.597000,1.568687,0.363000,0.123833
2,1.568800,1.562537,0.366500,0.137390
3,1.565800,1.560395,0.372500,0.142119


/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


### Обучение модели в режиме PEFT с использованием LoRA.

* lora_alpha: Значение 32 дает стабильное обучение, лучше, чем 16/64
* r: Значение 8 дает хорошее качество при минимальной нагрузке по памяти и числу параметров. При r=16 качество росло незначительно, а ресурсы расходовались почти вдвое больше.

In [19]:
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    inference_mode=False,
    r=8,                  # размерность low-rank матриц 
    lora_alpha=32,        # масштаб параметров LoRA
    lora_dropout=0.1      # Dropout на LoRA-слоях
)

model_lora = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=NUM_LABELS)
results_lora = train_model(model_lora, tokenized_datasets["train"], tokenized_datasets["validation"], name="lora_tuning", peft_cfg=lora_config)


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
No label_names provided for model class `PeftModelForSequenceClassification`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,1.398800,1.087106,0.587500,0.262615
2,1.004400,0.888184,0.675000,0.426045
3,0.882000,0.833496,0.682500,0.437521


/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


### Выводим метрики

In [22]:
# Compile Results
results_df = pd.DataFrame([
    {"method": "Full Finetuning", **results_full},
    {"method": "Linear Probing", **results_linear},
    {"method": "Prefix Tuning", **results_prefix},
    {"method": "LoRA (r=8)", **results_lora},
])

print(results_df)
results_df.to_csv("experiment_results.csv", index=False)

            method  eval_loss  eval_accuracy   eval_f1  eval_runtime  \
0  Full Finetuning   0.150185         0.9390  0.911826        9.6740   
1   Linear Probing   1.554026         0.3930  0.150307        9.6885   
2    Prefix Tuning   1.560395         0.3725  0.142119       10.0294   
3       LoRA (r=8)   0.833496         0.6825  0.437521       10.1395   

   eval_samples_per_second  eval_steps_per_second  epoch  trainable_params  \
0                  206.740                  6.512    3.0         109486854   
1                  206.431                  6.503    3.0              4614   
2                  199.413                  6.282    3.0            373254   
3                  197.248                  6.213    3.0            299526   

   total_params  training_time_min  gpu_memory_mb  
0     109486854          11.693431    5462.787598  
1     109486854           5.666015    4204.981934  
2     109860108           8.964644    4622.698730  
3     109786380           9.196866    46

### Вывод:

Видно, что Full Finetuning показывает лучшие результаты по всем метрикам качества.

LoRA — единственный метод PEFT, показывающий неплохую точность.

Prefix и Linear probing работают очень слабо — вероятно, классификационная задача слишком сложна для адаптации через небольшое число параметров.



Однако, разница во времени обучения есть. Например, Linear Probing почти в 2 раза быстрее, чем Full Finetuning.

In [17]:
def train_model_with_callbacks(model, train_data, eval_data, name, peft_cfg=None):
    if peft_cfg:
        model = get_peft_model(model, peft_cfg)

    training_args = TrainingArguments(
        output_dir=f"./results/{name}",
        eval_strategy="epoch",
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        num_train_epochs=EPOCHS,
        learning_rate=2e-5,
        weight_decay=0.05,
        seed=seed,
        logging_dir=f"./logs/{name}",
        report_to="none",
        metric_for_best_model="eval_loss",
        greater_is_better=False
    )

    early_stopping = EarlyStoppingCallback(early_stopping_patience=2)

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_data,
        eval_dataset=eval_data,
        compute_metrics=compute_metrics,
        callbacks=[early_stopping]
    )

    start = time.time()
    trainer.train()
    end = time.time()

    results = trainer.evaluate()
    results.update({
        "trainable_params": sum(p.numel() for p in model.parameters() if p.requires_grad),
        "total_params": sum(p.numel() for p in model.parameters()),
        "training_time_min": (end - start) / 60,
        "gpu_memory_mb": torch.cuda.max_memory_allocated() / 1024**2
    })
    return results

# Создаём модель
model_full = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, 
    num_labels=NUM_LABELS
)

# Обучаем с callbacks
results_full = train_model_with_callbacks(
    model_full,
    tokenized_datasets["train"],
    tokenized_datasets["validation"],
    name="full_finetuning_reg"
)


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Using EarlyStoppingCallback without load_best_model_at_end=True. Once training is finished, the best model will not be loaded automatically.
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.616500,0.218677,0.927000,0.903169
2,0.157100,0.171141,0.934000,0.907795
3,0.100800,0.167842,0.934000,0.907349


/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked t